# Python Iterables and Iterators: A Practical Guide

## Core Concepts

### The Book and Finger Analogy
Think of it this way:
- **The BOOK** = Iterable (you CAN read through it)
- **Your FINGER marking where you are** = Iterator (it DOES the reading)

The book itself doesn't know where you're reading. But your finger:
1. Knows which word you're on
2. Can move to the next word  
3. Knows when you've reached the end

In [ ]:
# A list is like a book - it just sits there with data
my_list = [10, 20, 30]

# An iterator is like your reading finger - a separate object!
reading_finger = iter(my_list)
print(type(my_list))          # <class 'list'>
print(type(reading_finger))   # <class 'list_iterator'> - Different object!

### Why Iterators Are Separate Objects
If the list itself kept track of position, nested loops would break:

In [ ]:
my_list = [10, 20, 30]

# This works because each loop gets its own iterator (reading finger)
for x in my_list:
    for y in my_list:  # Different finger, different position!
        print(x, y)    # Prints all 9 combinations

### The Two Key Principles
1. **Iterables**: Anything you CAN loop through (lists, strings, dicts, ranges, etc.)
2. **Iterators**: The object that DOES the looping, tracking position

### Visual Mental Model - The "next" Pointer
Think of the iterator as having a pointer that moves through your data:

```
Initial state:     [>] 1, 2, 3     (iterator created, pointing "before" start)
After 1st next():   1 [>] 2, 3     (returned 1, now pointing between 1 and 2)
After 2nd next():   1, 2 [>] 3     (returned 2, now pointing between 2 and 3)
After 3rd next():   1, 2, 3 [>]    (returned 3, now at the end)
```

The `next()` function:
- Returns what the pointer is currently looking at
- Moves the pointer forward to the next position
- Raises `StopIteration` when there's nothing left

## Basic Examples

### What Python Does Behind the Scenes

In [ ]:
# What you write:
for item in [1, 2, 3]:
    print(item)

# What Python actually does:
my_list = [1, 2, 3]
my_iterator = iter(my_list)  # Create iterator from iterable
while True:
    try:
        item = next(my_iterator)  # Get next item
        print(item)
    except StopIteration:         # No more items!
        break

### Manual Iterator Control

In [ ]:
numbers = [1, 2, 3]
it = iter(numbers)

print(next(it))  # 1
print(next(it))  # 2
print(next(it))  # 3
# next(it)       # StopIteration error!

### Multiple Iterators = Independent Positions

In [ ]:
numbers = [1, 2, 3]
iter1 = iter(numbers)
iter2 = iter(numbers)

print(next(iter1))  # 1
print(next(iter2))  # 1 (independent position!)
print(next(iter1))  # 2

## What Different Iterables Give You

### The Golden Rule
**Different iterables yield different things when iterated:**

In [ ]:
# Lists give elements
for item in [1, 2, 3]:
    print(item)  # 1, 2, 3

# Strings give characters
for char in "hello":
    print(char)  # h, e, l, l, o

# Dictionaries give keys (by default!)
for key in {"a": 1, "b": 2}:
    print(key)  # a, b

# Ranges give numbers
for num in range(3):
    print(num)  # 0, 1, 2

### Dictionary Iteration Options

In [ ]:
my_dict = {"a": 1, "b": 2}

# Default: keys only
list(my_dict)                # ['a', 'b']

# Explicit options
list(my_dict.keys())         # ['a', 'b']
list(my_dict.values())       # [1, 2]
list(my_dict.items())        # [('a', 1), ('b', 2)]

# That's why this unpacking works:
for key, value in my_dict.items():
    print(f"{key}: {value}")  # Each item is a tuple!

## Iterator Memory Efficiency

### The Problem with Eager Evaluation

In [ ]:
# Bad: Creates entire list in memory
numbers = list(range(1000000))  # Million numbers in RAM!
for n in numbers:
    if n > 5:
        break  # Wasted 999,994 numbers!

# Good: Creates values on demand
numbers = range(1000000)  # Iterator - almost no memory
for n in numbers:
    if n > 5:
        break  # Only created 6 numbers!

### Map, Filter, and Zip Return Iterators

In [ ]:
# These all return iterators (lazy evaluation)
numbers = range(1000000)
squared = map(lambda x: x**2, numbers)        # No computation yet!
evens = filter(lambda x: x % 2 == 0, numbers) # Still no computation!

# Computation happens only when needed
first_squared = next(squared)  # NOW it calculates 0**2


### The Iterator Exhaustion Trap

In [ ]:
def square_with_print(x):
    print(f"Squaring {x}")
    return x ** 2

numbers = range(10)
squared = map(square_with_print, numbers)

# BAD: This squares ALL numbers, then slices
first_three = list(squared)[:3]  # Prints "Squaring" 10 times!

# GOOD: Use itertools.islice
from itertools import islice
squared = map(square_with_print, range(10))
first_three = list(islice(squared, 3))  # Prints "Squaring" only 3 times!

## Edge Cases and Gotchas

### 1. Iterators Are One-Time Use

In [ ]:
numbers = [1, 2, 3]
it = iter(numbers)
list(it)  # [1, 2, 3]
list(it)  # [] - Empty! Iterator is exhausted

### 2. Partially Consumed Iterators

In [ ]:
it = iter([1, 2, 3, 4, 5])
print(next(it))  # 1
print(next(it))  # 2

# for loop continues from current position!
for num in it:
    print(num)  # 3, 4, 5 (not starting over)

### 3. Strings Are Iterables Too

In [ ]:
def process_items(items):
    for item in items:
        print(f"Processing: {item}")

process_items([1, 2, 3])     # Works as expected
process_items("hello")       # Processes each character!

### 4. Empty Iterables

In [ ]:
for item in []:
    print("This never runs")  # No error, just skips

it = iter([])
# next(it)  # StopIteration immediately!

### 5. Infinite Iterators

In [ ]:
from itertools import count

for i in count():  # Starts at 0, never stops!
    print(i)
    if i > 5:
        break  # Always have an exit condition!

## Expert Tips

### 1. Check If Something Is Iterable

In [ ]:
def is_iterable(obj):
    try:
        iter(obj)
        return True
    except TypeError:
        return False

is_iterable([1, 2, 3])  # True
is_iterable(42)         # False

### 2. Create Your Own Iterator


In [ ]:
class Counter:
    def __init__(self, start, end):
        self.current = start
        self.end = end
    
    def __iter__(self):
        return self
    
    def __next__(self):
        if self.current < self.end:
            num = self.current
            self.current += 1
            return num
        raise StopIteration

# Use it like any iterator
for i in Counter(1, 4):
    print(i)  # 1, 2, 3

### 3. Chain Operations Efficiently


In [ ]:
# Memory efficient pipeline
numbers = range(1000000)
result = (x**2 for x in numbers if x % 2 == 0)  # Generator expression
first_ten = list(islice(result, 10))  # Only processes what's needed

### 4. Use `enumerate` for Index + Value

In [ ]:
fruits = ['apple', 'banana', 'orange']
for i, fruit in enumerate(fruits):
    print(f"{i}: {fruit}")  # 0: apple, 1: banana, 2: orange

### 5. Zip Stops at Shortest Iterable

In [ ]:
list(zip([1, 2, 3], ['a', 'b']))  # [(1, 'a'), (2, 'b')]
# Note: 3 is dropped!

## Common Patterns

### Reading Large Files

In [ ]:
# Bad: Loads entire file
lines = open('huge_file.txt').readlines()

# Good: Iterator over lines
with open('huge_file.txt') as f:
    for line in f:  # f is an iterator!
        process(line)

### Testing Iterator Contents


In [ ]:
# Peek at first item without consuming entire iterator
from itertools import tee

it1, it2 = tee(original_iterator)
first_item = next(it1)  # Check first item
# it2 still has all items

### Converting Between Types


In [ ]:
# Any iterable can become a list/tuple/set
my_list = list("hello")           # ['h', 'e', 'l', 'l', 'o']
my_tuple = tuple(range(3))        # (0, 1, 2)
my_set = set([1, 2, 2, 3])        # {1, 2, 3}

## Cautions

### ⚠️ Don't Modify While Iterating


In [ ]:
# BAD - Unpredictable behavior
my_list = [1, 2, 3, 4, 5]
for item in my_list:
    if item % 2 == 0:
        my_list.remove(item)  # DON'T DO THIS!

# GOOD - Create new list
my_list = [1, 2, 3, 4, 5]
my_list = [item for item in my_list if item % 2 != 0]

### ⚠️ Iterator != Iterable


In [ ]:
numbers = [1, 2, 3]
it = iter(numbers)

# This works
for n in numbers:  # numbers is iterable
    print(n)

# This also works but exhausts iterator
for n in it:       # it is iterator (also iterable)
    print(n)

# But this fails
for n in it:       # it is now exhausted!
    print(n)       # Nothing prints

### ⚠️ Some Operations Consume Entire Iterator


In [ ]:
it = iter(range(1000000))
length = len(list(it))  # Converts entire iterator to list!
# it is now exhausted AND you used lots of memory

## Quick Reference

| Function | What it needs | What it returns | Note |
|----------|--------------|-----------------|------|
| `list()` | Any iterable | New list | Consumes entire iterator |
| `tuple()` | Any iterable | New tuple | Consumes entire iterator |
| `set()` | Any iterable | New set | Removes duplicates |
| `sum()` | Iterable of numbers | Single number | Must be numeric |
| `max()`/`min()` | Any iterable | Single item | Items must be comparable |
| `sorted()` | Any iterable | New list | Always returns list |
| `map()` | Function + iterable(s) | Iterator | Lazy evaluation |
| `filter()` | Function + iterable | Iterator | Lazy evaluation |
| `zip()` | Multiple iterables | Iterator | Stops at shortest |

## Summary

Remember the two golden rules:
1. **You can pass any iterable** where Python expects one
2. **Think about what you'll get** when iterating over that type

Master these concepts and you'll write more efficient, Pythonic code!